# Module 1: Stationarity Tested, Not Eyeballed

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Beginner [Topic 17](../../Beginner/Topic_17_Stationarity.md) asked whether a
series holds still, and answered by looking at it. That works for the obvious
cases and fails for everything else, which is most of what you will meet.

This module replaces the eye with two tests, explains why you need **both**,
and shows what it costs to difference a series more than it needs.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

## 2. Two tests with opposite nulls

This is the part that trips people up, so it is worth stating plainly.

| Test | Null hypothesis | A small p value means |
|---|---|---|
| **ADF**, augmented Dickey Fuller | the series has a unit root, so it is **not** stationary | evidence **for** stationarity |
| **KPSS** | the series **is** stationary | evidence **against** stationarity |

They point in opposite directions. Running only one of them gives you half an
answer, and which half depends on which test you happened to choose.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss


def stationarity(s, name=""):
    """Both tests, and the verdict the pair implies."""
    a_p = adfuller(s.dropna(), autolag="AIC")[1]
    k_p = kpss(s.dropna(), regression="c", nlags="auto")[1]

    adf_rejects = a_p < 0.05
    kpss_rejects = k_p < 0.05
    if adf_rejects and not kpss_rejects:
        verdict = "stationary"
    elif not adf_rejects and kpss_rejects:
        verdict = "not stationary"
    elif adf_rejects and kpss_rejects:
        verdict = "conflict"
    else:
        verdict = "inconclusive"
    return {"series": name, "ADF p": round(a_p, 3), "KPSS p": round(k_p, 3),
            "verdict": verdict}

**One practical detail before the results.** KPSS p values come from a lookup
table that stops at 0.01 and 0.10. A reported 0.010 means "0.01 or less" and a
reported 0.100 means "0.10 or more". Report them as bounds, not as numbers, or
you will find yourself comparing a 0.100 against a 0.094 as though the
difference meant something.

## 3. Three agencies, three different answers

In [ ]:
rows = [stationarity(rate("A008"), "Lakeshore County, rate"),
        stationarity(rate("A012"), "Ashfell, rate"),
        stationarity(rate("A007"), "Summit County, rate"),
        stationarity(counts("A012"), "Ashfell, counts")]
pd.DataFrame(rows).set_index("series")

Three of the four verdicts appear in one table.

**Lakeshore County is stationary.** Both tests agree, which matches what
Beginner Topic 17 claimed from the chart alone: this agency has been bouncing
around the same level for seven years.

**Summit County is not stationary.** Both tests agree again. It is the agency
the dataset built with a decline of about 12 percent a year, so this is the
answer the tests are supposed to give.

**Ashfell's counts are inconclusive.** Neither test rejects. That is the fourth
verdict, and it means exactly what it says: **there is not enough evidence to
decide**, which is different from concluding the series is stationary.

## 4. The count and the rate can disagree

Look again at the two Ashfell rows. The **rate** tests as not stationary. The
**counts** are inconclusive.

Nothing is inconsistent here. Intermediate
[Module 3](../../Intermediate/Module_03_Choosing_A_Denominator.md) showed that
a rate's trend is the count's trend minus the denominator's. Ashfell's arrests
grew about 1.3 percent a year, which partly offsets the fall in incidents, so
the count has a weaker trend than the rate does and the tests can see it less
clearly.

In [ ]:
import statsmodels.formula.api as smf

g = final[(final["agency_id"] == "A012") & (final["year_month"] <= "2025-12")].copy()
idx = pd.PeriodIndex(g["year_month"], freq="M")
g["t"] = (idx.year - 2019) * 12 + idx.month - 1
g["rate"] = 100 * g["n_uof"] / g["n_arrests"]

per_year = lambda b: 100 * (np.exp(12 * b) - 1)
for col in ["n_uof", "n_arrests", "rate"]:
    fit = smf.ols(f"np.log({col}) ~ t", data=g).fit()
    print(f"  {col:10s} {per_year(fit.params['t']):+6.2f} percent a year")

**Which one should you test?** Whichever one you intend to model. The verdict
is a property of the series in front of you, not of the agency.

## 5. Making a series stationary, and going too far

The usual fix is differencing: model the change rather than the level. The
usual mistake is doing it more than once when once was enough.

Over differencing has a signature you can check for. It **increases the
variance** of what is left, and it drives the lag one autocorrelation toward
**minus one half**.

In [ ]:
s = np.log(rate("A007"))        # Summit County, the strongly trending one

transforms = {
    "level": s,
    "first difference": s.diff(),
    "twice differenced": s.diff().diff(),
    "seasonal difference": s.diff(12),
    "first and seasonal": s.diff().diff(12),
}
pd.DataFrame([{"transform": k,
               "observations left": v.dropna().shape[0],
               "variance": round(v.dropna().var(), 4),
               "lag one autocorrelation": round(v.dropna().autocorr(1), 2)}
              for k, v in transforms.items()]).set_index("transform")

Read the variance column. The level is 0.195. One difference brings it down to
0.138, which is the point of differencing. **Two differences push it back up to
0.410, worse than doing nothing at all**, and the lag one autocorrelation falls
to minus 0.68.

Now read the last two rows. A **seasonal difference alone** gives the lowest
variance of any transform, 0.115, with a lag one autocorrelation of minus 0.03,
which is about as close to white noise as this series gets. Adding a first
difference on top of it takes the variance back up to 0.235.

**For this series the seasonal difference alone is the right answer**, and the
combination most software applies by default is over differencing.

## 6. Check that the transform worked

In [ ]:
pd.DataFrame([
    stationarity(np.log(rate("A007")).diff(), "Summit County, log rate, differenced"),
    stationarity(np.log(rate("A012")).diff(), "Ashfell, log rate, differenced"),
]).set_index("series")

Both are now stationary on both tests, which is the state the models in
[Module 4](Module_04_ARIMA_End_To_End.ipynb) onward assume.

## 7. What the tests cannot do

| Limitation | Why it matters |
|---|---|
| They are weak in short series | ninety months is not much; inconclusive is common and honest |
| A structural break looks like a unit root | a level shift will fail the tests for the wrong reason |
| Seasonality confuses them | test the deseasonalised series, or use a seasonal difference first |
| They say nothing about which transform to use | the tests diagnose, the variance and autocorrelation choose |
| Passing is not a licence | a stationary series can still have outliers, changing variance, or a break |

The last row deserves its own sentence. **Stationarity is one assumption among
several.** Beginner [Topic 11](../../Beginner/Topic_11_Outliers_And_Spikes.md)
and Intermediate [Module 8](../../Intermediate/Module_08_Rolling_Statistics_And_Control_Limits.md)
check different things, and a series can pass here and fail there.

## Exercise

Test Tarnbridge, which contains one documented week of civil unrest in June
2021. Does the outlier change the verdict?

In [ ]:
# Fill in the blanks, then run.
AGENCY = None              # try "A002"
DROP_THE_EVENT = None      # try False, then True

if AGENCY is not None and DROP_THE_EVENT is not None:
    s = rate(AGENCY)
    if DROP_THE_EVENT:
        s = s[~((s.index.year == 2021) & (s.index.month == 6))]
    label = "without the unrest month" if DROP_THE_EVENT else "with the unrest month"
    print(pd.DataFrame([stationarity(s, label)]).set_index("series").to_string())
else:
    print("Set AGENCY and DROP_THE_EVENT above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A002"
DROP_THE_EVENT = False      # then True
```

The verdict does not change, and that is worth knowing rather than assuming.
A single extreme observation is not a unit root and not a trend, so neither
test has much reason to respond to it. What the outlier does affect is
everything downstream: the estimated variance, the fitted coefficients, and the
width of every interval.

The general point is the one in the table above. **Stationarity tests answer
one question.** Passing them tells you the level is not wandering. It tells you
nothing about whether the series contains an event that will distort every
model you fit to it, and you have to check for that separately.

</details>

---

**Next:** [Module 2, Incident Counts Are Not Gaussian](Module_02_Counts_Are_Not_Gaussian.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*